# 00 · Kernel check

## What this checks

This notebook checks nothing about Fieldwork. It answers one question: **does this kernel really have the packages
the checks need, and which versions?**

A kernel is the Python that runs these notebooks. In the browser it is Pyodide, a Python built to run inside a web
page. Its packages can differ from a Python installed on a computer.

## How it is checked

A list of packages says what should be there. Only loading them shows what is there. So the cells below ask for nine
packages and print the version of each one they find.

Keep this output with any result you get in this kernel. It says which versions produced it.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
NEEDED = ["pyproj", "shapely", "rasterio", "geopandas", "fiona", "numpy", "pandas", "h3", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    print("Falling back to piplite for anything still missing.")
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import importlib, platform, sys

print(f"runtime   CPython {platform.python_version()} on {platform.system()} {platform.machine()}")
print(f"pyodide   {'yes' if 'pyodide' in sys.modules or platform.system() == 'Emscripten' else 'no'}")
print()
for name in ["pyproj", "shapely", "rasterio", "geopandas", "fiona", "numpy", "pandas", "h3", "scipy"]:
    try:
        module = importlib.import_module(name)
        print(f"{name:11s} {getattr(module, '__version__', 'unknown')}")
    except Exception as error:
        print(f"{name:11s} ABSENT ({type(error).__name__}: {error})")

In [ ]:
# PROJ and GEOS carry their own versions, which are what the comparisons actually exercise.
import pyproj
print("PROJ", pyproj.proj_version_str)
print("pyproj data dir", pyproj.datadir.get_data_dir())
try:
    import shapely
    print("GEOS", shapely.geos_version_string)
except Exception as error:
    print("GEOS unavailable:", error)

## What this does not show

**That an import works.** A package can import and still fail when it is used.

- pyproj can import without its PROJ data folder. It then fails on any conversion that needs a grid file.
- The checks in this lab change projection on one datum only, so they need no grid file.
- A datum shift would need one. It must then fail with an error rather than download one silently.

**That the packages can be used together.** This was measured in this kernel on 11 October 2026. Each trial
started a fresh kernel, asked for a set of packages, then opened this lab's small GeoTIFF (24 by 18 cells) with
`rasterio.open`.

| Requested | Trials | `rasterio.open` |
| --- | --- | --- |
| `pyproj`, `rasterio` — either order | 2 | **fatal** |
| `numpy, rasterio, scipy, shapely` **+ `pyproj`** | 1 | **fatal** |
| the nine below, in the order below | 4 | **fatal** |
| `fiona` **before** `rasterio` | 1 | **fatal** |
| `numpy, rasterio, scipy, shapely` | 3 | ok |
| the four **+ `fiona` after** `rasterio` | 1 | ok |
| the nine reordered, `rasterio` early and `pyproj` last | 3 | ok |

"Fatal" means `RuntimeError: null function` inside GDAL, the raster library, as built for the browser. It **shuts the
kernel down instead of reporting an error**. No cell shows an error. So a runner waiting for a cell's output waits
for ever. This cost 28 minutes on notebook 02 before the browser runner learned to watch for a dead kernel.

The table supports two readings:

- pyproj and rasterio cannot share a kernel.
- They can, if fiona is also loaded after rasterio.

That is why notebook 02 asks only for the four packages it uses.

**Why this happens.** Three explanations were tried, and the next measurement ruled out each one:

- **Downloads.** It is not the time taken to download packages: this notebook loads all nine in 15 seconds.
- **fiona and rasterio.** It is not a clash between them over GDAL: fiona loaded after rasterio is fine.
- **Load order.** It is not load order alone: pyproj crashes in both orders.

So the table is a map of what works, not an explanation. The fault is in this Pyodide release, not in anything the lab
controls.

**The crash itself.** This notebook does not show the crash, because that would shut the kernel down and leave it
unable to report. It still asks for all nine packages, because its job is to report what the browser's Python
provides. Having a package and being able to use it are different claims.